<a href="https://colab.research.google.com/github/Karimkhab/introduction-to-machine-learning/blob/main/%D0%9A%D0%BE%D0%BF%D0%B8%D1%8F_%D0%B1%D0%BB%D0%BE%D0%BA%D0%BD%D0%BE%D1%82%D0%B0_%22NLP_HW_Lab01_Poetry_generation_v5_ipynb%22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Генерация поэзии с помощью нейронных сетей: шаг 1
##### Автор: [Радослав Нейчев](https://www.linkedin.com/in/radoslav-neychev/), @neychev

Ваша основная задача: научиться генерироват стихи с помощью простой рекуррентной нейронной сети (Vanilla RNN). В качестве корпуса текстов для обучения будет выступать роман в стихах "Евгений Онегин" Александра Сергеевича Пушкина.

In [85]:
# do not change the code in the block below
# __________start of block__________
import string
import os
from random import sample

import numpy as np
import torch, torch.nn as nn
import torch.nn.functional as F

from IPython.display import clear_output

import matplotlib.pyplot as plt
# __________end of block__________

In [86]:
# do not change the code in the block below
# __________start of block__________
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
print('{} device is available'.format(device))
# __________end of block__________

cuda device is available


#### 1. Загрузка данных.

In [87]:
# do not change the code in the block below
# __________start of block__________
!wget https://raw.githubusercontent.com/neychev/small_DL_repo/master/datasets/onegin.txt

with open('onegin.txt', 'r') as iofile:
    text = iofile.readlines()

text = "".join([x.replace('\t\t', '').lower() for x in text])
# __________end of block__________

--2026-03-06 20:02:11--  https://raw.githubusercontent.com/neychev/small_DL_repo/master/datasets/onegin.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 262521 (256K) [text/plain]
Saving to: ‘onegin.txt.3’

onegin.txt.3        100%[===================>] 256.37K  --.-KB/s    in 0.003s  

2026-03-06 20:02:11 (73.7 MB/s) - ‘onegin.txt.3’ saved [262521/262521]



#### 2. Построение словаря и предобработка текста
В данном задании требуется построить языковую модель на уровне символов. Приведем весь текст к нижнему регистру и построим словарь из всех символов в доступном корпусе текстов. Также добавим токен `<sos>`.

In [88]:
# do not change the code in the block below
# __________start of block__________
tokens = sorted(set(text.lower())) + ['<sos>']
num_tokens = len(tokens)

assert num_tokens == 84, "Check the tokenization process"

token_to_idx = {x: idx for idx, x in enumerate(tokens)}
idx_to_token = {idx: x for idx, x in enumerate(tokens)}

assert len(tokens) == len(token_to_idx), "Mapping should be unique"

print("Seems fine!")


text_encoded = [token_to_idx[x] for x in text]
# __________end of block__________

Seems fine!


__Ваша задача__: обучить классическую рекуррентную нейронную сеть (Vanilla RNN) предсказывать следующий символ на полученном корпусе текстов и сгенерировать последовательность длины 100 для фиксированной начальной фразы.

Вы можете воспользоваться кодом с занятие №6 или же обратиться к следующим ссылкам:
* Замечательная статья за авторством Andrej Karpathy об использовании RNN: [link](http://karpathy.github.io/2015/05/21/rnn-effectiveness/)
* Пример char-rnn от Andrej Karpathy: [github repo](https://github.com/karpathy/char-rnn)
* Замечательный пример генерации поэзии Шекспира: [github repo](https://github.com/spro/practical-pytorch/blob/master/char-rnn-generation/char-rnn-generation.ipynb)

Данное задание является достаточно творческим. Не страшно, если поначалу оно вызывает затруднения. Последняя ссылка в списке выше может быть особенно полезна в данном случае.

Далее для вашего удобства реализована функция, которая генерирует случайный батч размера `batch_size` из строк длиной `seq_length`. Вы можете использовать его при обучении модели.

In [89]:
# do not change the code in the block below
# __________start of block__________
batch_size = 256
seq_length = 100
start_column = np.zeros((batch_size, 1), dtype=int) + token_to_idx['<sos>']

def generate_chunk():
    global text_encoded, start_column, batch_size, seq_length

    start_index = np.random.randint(0, len(text_encoded) - batch_size*seq_length - 1)
    data = np.array(text_encoded[start_index:start_index + batch_size*seq_length]).reshape((batch_size, -1))
    yield np.hstack((start_column, data))
# __________end of block__________

Пример батча:

In [90]:
idx = next(generate_chunk())
idx

array([[83, 50, 47, ..., 46, 50, 48],
       [83, 58, 50, ..., 59, 50, 47],
       [83, 59, 56, ..., 54,  1, 68],
       ...,
       [83, 73, 76, ...,  1, 63, 61],
       [83, 50, 60, ..., 76,  1, 61],
       [83, 45, 49, ..., 64, 62, 63]])

In [91]:
next(generate_chunk())

array([[83, 59, 54, ..., 59, 55, 50],
       [83, 63, 55, ..., 47, 59, 61],
       [83, 53, 63, ..., 51, 49, 59],
       ...,
       [83, 73, 13, ..., 55, 50,  7],
       [83,  0, 47, ...,  1, 60, 45],
       [83, 56, 73, ..., 58, 72,  1]])

Далее вам предстоит написать код для обучения модели и генерации текста.

In [92]:
# your beautiful experiments here
# Model
class VanillaRNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, num_layers=1):
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.output_size = output_size
        self.num_layers = num_layers

        self.encoder = nn.Embedding(input_size, hidden_size)
        self.gru = nn.GRU(hidden_size, hidden_size, num_layers)
        self.decoder = nn.Linear(hidden_size, output_size)

    def forward(self, input, hidden):
        # input: (batch_size,)
        input = self.encoder(input)          # (batch_size, hidden_size)
        input = input.unsqueeze(0)           # (1, batch_size, hidden_size)

        output, hidden = self.gru(input, hidden)   # output: (1, batch_size, hidden_size)
        output = self.decoder(output.squeeze(0))   # (batch_size, output_size)

        return output, hidden

    def init_hidden(self, batch_size):
        return torch.zeros(self.num_layers, batch_size, self.hidden_size)

В качестве иллюстрации ниже доступен график значений функции потерь, построенный в ходе обучения авторской сети (сам код для ее обучения вам и предстоит написать).

In [93]:
from tqdm.notebook import tqdm

In [94]:
torch.cuda.is_available()

True

In [95]:
INPUT_SIZE, OUTPUT_SIZE = num_tokens, num_tokens
HIDDEN_SIZE = 128

LEARNING_RATE = 1e-4

model = VanillaRNN(INPUT_SIZE, HIDDEN_SIZE, OUTPUT_SIZE)
optimizer = torch.optim.Adam(model.parameters(), lr = LEARNING_RATE)
criterion = nn.CrossEntropyLoss()
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

In [96]:
def random_training_set():
    chunk = next(generate_chunk())
    input = torch.from_numpy(chunk[:,:-1]).long()
    target = torch.from_numpy(chunk[:,1:]).long()
    return input, target

In [100]:
def train(inputs, targets, device):
    model.to(device)
    inputs = inputs.transpose(0, 1).to(device)   # (seq_len, batch_size)
    targets = targets.transpose(0, 1).to(device)  # (seq_len, batch_size)

    hidden = model.init_hidden(inputs.shape[1]).to(device)  # batch_size
    model.zero_grad()
    loss = 0

    for input_t, target_t in zip(inputs, targets):
        output, hidden = model(input_t, hidden)   # input_t: (batch_size,)
        loss += criterion(output, target_t)       # output: (batch_size, vocab_size)

    loss.backward()
    optimizer.step()

    return loss.item() / len(inputs)

In [102]:
def train_epoch(model, optimizer, criterion, device, number_epoch = 3000):
    loss_history = []
    pbar = tqdm(range(number_epoch), desc="Training")
    for epoch in pbar:
        loss = train(*random_training_set(), device)
        loss_history.append(loss)

        if epoch % 10 == 0:
            pbar.set_postfix({"loss": f"{loss:.4f}"})
    return loss_history

In [103]:
loss_history = train_epoch(model, optimizer, criterion, device)

Training:   0%|          | 0/3000 [00:00<?, ?it/s]

Шаблон функции `generate_sample` также доступен ниже. Вы можете как дозаполнить его, так и написать свою собственную функцию с нуля. Не забывайте, что все примеры в обучающей выборке начинались с токена `<sos>`.

In [111]:
def generate_sample(char_rnn, seed_phrase=None, max_length=500, temperature=1.0, device=device):
    char_rnn.eval()

    if seed_phrase is not None:
        x_sequence = [token_to_idx['<sos>']] + [token_to_idx[token] for token in seed_phrase]
    else:
        x_sequence = [token_to_idx['<sos>']]

    hidden = char_rnn.init_hidden(batch_size=1).to(device)

    with torch.no_grad():
        # прогреваем hidden на seed-фразе
        for token_ix in x_sequence[:-1]:
            x_tensor = torch.tensor([token_ix], dtype=torch.int64).to(device)
            _, hidden = char_rnn(x_tensor, hidden)

        last_token = x_sequence[-1]

        # хотим, чтобы после удаления <sos> осталось ровно max_length символов
        while len(x_sequence) - 1 < max_length:
            x_tensor = torch.tensor([last_token], dtype=torch.int64).to(device)
            logits, hidden = char_rnn(x_tensor, hidden)

            logits = logits / temperature
            probs = torch.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs[0], 1).item()

            x_sequence.append(next_token)
            last_token = next_token

    return ''.join(tokens[ix] for ix in x_sequence[1:])

Пример текста сгенерированного обученной моделью доступен ниже. Не страшно, что в тексте много несуществующих слов. Используемая модель очень проста: это простая классическая RNN.

In [120]:
print(generate_sample(model, 'да я тип крутые типы да реально кайф', max_length=500, temperature=0.8))

да я тип крутые типы да реально кайфи
предушала в не гомо,
едо в жен в не потом, не сердужной
доборик она взорост,
не вет! мено смем,
тороду молный русски разденной
перед не вся в ее чавон любовзыхо,
в на столеменье дуэла,
и пословили я вдной…
меня старунчет блязних поэта:
завоет и воковы
мно верекдот все печась поэт,
не на свот розаталон,
подыму соперчивой моя,
на томен нет; завор, их дальной году,
о слуша своей вас задах поется,
и стручала взгласем
от паще междела будки, не сероний,
предвордал


### Сдача задания
Сгенерируйте десять последовательностей длиной 500, используя строку ' мой дядя самых честных правил'. Температуру для генерации выберите самостоятельно на основании визуального качества генериуремого текста. Не забудьте удалить все технические токены в случае их наличия.

Сгенерированную последовательность сохрание в переменную `generated_phrase` и сдайте сгенерированный ниже файл в контест.

In [113]:
seed_phrase = ' мой дядя самых честных правил'

In [115]:
# generated_phrases = # your code here

# For example:

generated_phrases = [
    generate_sample(
        model,
        ' мой дядя самых честных правил',
        max_length=500,
        temperature=1.
    ).replace('<sos>', '')
    for _ in range(10)
]
generated_phrases


[' мой дядя самых честных правил:\nи в янула сверкой болтым?\nлюбовый ностало;\nи облаждо страмией… дедом,\nот свит беж отор для моей;\n«чаты сертвенье, будак!\nв неювшихи как нечнох стелел\nонего прошь ноный! —\nстатьяна лукое грина,\nдвет не копретила, герет,\nпогрудобе гламах.\n\n\n\nxxiii\n\nвот ниго щажет; любил ем\nчто жег уснятленьи заделя\nондетвенныв, его притил,\nдвони на фвилческоню\nне собом шумнове глушно благодать\nнето пистает оная мог,\nчувадел наем все дубые,\nлюбвит катая слибный;\nблегинов еродать, не ',
 ' мой дядя самых честных правил\nеще рден. онатая!\nчутьно дразноченьки мичей;\nсляже, верщиком изы!ась,\nкона стит, всё другое моевь;\nтатьто дровол вод торват. ец ей бользина\nв улым ней без оема модар, ких,\nей беду полгоу узововконную,\nв слеспятий свдит; безна буд,\nей шумненный мазу.\nи как из где услава славствах конен,\nдорога таня дотакий,\nто покниче ник безмне довер!\nию порад перей чериды\nна меж по– бутет, мачите; двед ольга\nно ей запиле точени.\nто ду

In [116]:
# do not change the code in the block below
# __________start of block__________

import json
if 'generated_phrases' not in locals():
    raise ValueError("Please, save generated phrases to `generated_phrases` variable")

for phrase in generated_phrases:

    if not isinstance(phrase, str):
        raise ValueError("The generated phrase should be a string")

    if len(phrase) != 500:
        raise ValueError("The `generated_phrase` length should be equal to 500")

    assert all([x in set(tokens) for x in set(list(phrase))]), 'Unknown tokens detected, check your submission!'


submission_dict = {
    'token_to_idx': token_to_idx,
    'generated_phrases': generated_phrases
}

with open('submission_dict.json', 'w') as iofile:
    json.dump(submission_dict, iofile)
print('File saved to `submission_dict.json`')
# __________end of block__________

File saved to `submission_dict.json`


На этом задание завершено. Поздравляем!